# Block 4 CPU smoke: segmentation provider

Техническая проверка открытого CPU-provider на процедурном изображении. Это не задание и не эталонное решение лабораторной DenseCRF.

In [ ]:
import importlib.util
import json
import random
import sys
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

def intersection_over_union(target, prediction):
    target = np.asarray(target, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    union = np.logical_or(target, prediction).sum()
    return 1.0 if union == 0 else float(np.logical_and(target, prediction).sum() / union)

def dice_score(target, prediction):
    target = np.asarray(target, dtype=bool)
    prediction = np.asarray(prediction, dtype=bool)
    denominator = target.sum() + prediction.sum()
    return 1.0 if denominator == 0 else float(2 * np.logical_and(target, prediction).sum() / denominator)

In [ ]:
provider_path = Path("../methodical-guidelines/students/lab_segmentation.py").resolve()
assert provider_path.is_file(), provider_path

spec = importlib.util.spec_from_file_location("block4_smoke_provider", provider_path)
assert spec is not None and spec.loader is not None
provider_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = provider_module
spec.loader.exec_module(provider_module)
SegmentationPredictor = provider_module.SegmentationPredictor

In [ ]:
with tempfile.TemporaryDirectory(prefix="block4-smoke-") as directory:
    root = Path(directory)
    image = np.zeros((64, 64, 3), dtype=np.uint8)
    target = np.zeros((64, 64), dtype=np.uint8)
    image[20:44, 20:44] = (240, 180, 60)
    target[20:44, 20:44] = 1
    image_path = root / "sample.png"
    Image.fromarray(image).save(image_path)

    result = SegmentationPredictor(threshold=0.55).predict(image_path)
    iou = intersection_over_union(target, result.mask)
    dice = dice_score(target, result.mask)

    assert result.probability.shape == target.shape
    assert result.mask.shape == target.shape
    assert np.isfinite(result.probability).all()
    assert 0.0 <= float(result.probability.min()) <= float(result.probability.max()) <= 1.0
    assert iou > 0.95 and dice > 0.95
    assert result.metadata["provider"] == "numpy-cpu"

    journal_path = root / "runs.jsonl"
    record = {
        "seed": SEED,
        "metrics": {"iou": iou, "dice": dice},
        "config": {"provider": "numpy-cpu", "threshold": 0.55},
        "tags": ["block4", "cpu-smoke"],
    }
    journal_path.write_text(json.dumps(record, sort_keys=True) + "\n", encoding="utf-8")
    persisted = json.loads(journal_path.read_text(encoding="utf-8").strip())
    assert persisted["metrics"]["iou"] == iou

print({"iou": iou, "dice": dice})
print("Block 4 CPU smoke: OK")